In [62]:
import requests

url = "https://api.coinpaprika.com/v1/tickers?quotes=USD"

response = requests.get(url)

response



<Response [200]>

In [63]:
response_json = response.json()

In [64]:
coin_list = response_json[:5]

In [65]:
coin_list

[{'id': 'btc-bitcoin',
  'name': 'Bitcoin',
  'symbol': 'BTC',
  'rank': 1,
  'total_supply': 20050291,
  'max_supply': 21000000,
  'beta_value': 0.969682,
  'first_data_at': '2010-07-17T00:00:00Z',
  'last_updated': '2026-06-30T17:15:16Z',
  'quotes': {'USD': {'price': 58322.90569032994,
    'volume_24h': 28018902003.997353,
    'volume_24h_change_24h': -5.809999942779541,
    'market_cap': 1169391056087,
    'market_cap_change_24h': -3.4700000286102295,
    'percent_change_15m': -0.20999999344348907,
    'percent_change_30m': -0.019999999552965164,
    'percent_change_1h': 0.12999999523162842,
    'percent_change_6h': -1.3700000047683716,
    'percent_change_12h': -1.649999976158142,
    'percent_change_24h': -3.4700000286102295,
    'percent_change_7d': -6.389999866485596,
    'percent_change_30d': 0,
    'percent_change_1y': 0,
    'ath_price': 126173.1777846797,
    'ath_date': '2025-10-06T19:00:40Z',
    'percent_from_price_ath': -53.7}}},
 {'id': 'eth-ethereum',
  'name': 'Ether

In [66]:
type(coin_list)

list

In [67]:
!pip install pandas


In [113]:
import pandas as pd

coinz_df = pd.DataFrame(coin_list)

coinz_df

,id,name,symbol,rank,total_supply,max_supply,beta_value,first_data_at,last_updated,quotes
0,btc-bitcoin,Bitcoin,BTC,1,20050291,21000000,0.969682,2010-07-17T00:00:00Z,2026-06-30T17:15:16Z,"{'USD': {'price': 58322.90569032994, 'volume_2..."
1,eth-ethereum,Ethereum,ETH,2,120232214,0,1.449370,2015-08-07T00:00:00Z,2026-06-30T17:15:16Z,"{'USD': {'price': 1568.5270173044314, 'volume_..."
2,usdt-tether,Tether,USDT,3,190457881824,0,-0.001709,2015-02-25T00:00:00Z,2026-06-30T17:15:16Z,"{'USD': {'price': 0.9985966544837946, 'volume_..."
3,bnb-binance-coin,BNB,BNB,4,139184442,200000000,1.049170,2017-07-25T00:00:00Z,2026-06-30T17:15:16Z,"{'USD': {'price': 545.9989955041905, 'volume_2..."
4,usdc-usd-coin,USDC,USDC,5,73604840553,0,-0.003661,2018-10-09T00:00:00Z,2026-06-30T17:15:16Z,"{'USD': {'price': 0.9997508068409844, 'volume_..."


In [114]:
coinz_df = coinz_df[[ 'name', 'symbol',  'total_supply', 'first_data_at', 'quotes' ]]
                     

In [115]:
coinz_df = coinz_df.rename(columns={ 'name': 'coin_name', 'first_data_at': 'invention_date'})

In [116]:
coinz_df["usd_price"] = coinz_df["quotes"].apply(lambda q: q["USD"]["price"])

In [117]:
coinz_df = coinz_df.drop(columns=['quotes'])

In [118]:
coinz_df["invention_date"] = pd.to_datetime(coinz_df["invention_date"], errors="coerce")

In [119]:
coinz_df["invention_date"] = pd.to_datetime(coinz_df["invention_date"], errors="coerce")

In [120]:
coinz_df["usd_price"] = coinz_df["usd_price"].round(4)

In [121]:
coinz_df["total_supply"] = coinz_df["total_supply"].map("{:,.0f}".format)

In [123]:
coinz_df

,coin_name,symbol,total_supply,invention_date,usd_price
0,Bitcoin,BTC,"20,050,291",2010-07-17,58322.9057
1,Ethereum,ETH,"120,232,214",2015-08-07,1568.5270
2,Tether,USDT,"190,457,881,824",2015-02-25,0.9986
3,BNB,BNB,"139,184,442",2017-07-25,545.9990
4,USDC,USDC,"73,604,840,553",2018-10-09,0.9998


In [131]:
import os

from sqlalchemy import create_engine, text


DATABASE_NAME = os.getenv('DATABASE_NAME')
DATABASE_USER = os.getenv('DATABASE_USER')
DATABASE_PORT = os.getenv('DATABASE_PORT')
DATABASE_PASSWORD = os.getenv('DATABASE_PASSWORD')
DATABASE_HOST = os.getenv('DATABASE_HOST')    

try:
    engine = create_engine(
        f'postgresql://{DATABASE_USER}:{DATABASE_PASSWORD}@{DATABASE_HOST}:{DATABASE_PORT}/{DATABASE_NAME}'
    )

    # Load DataFrame to PostgreSQL
    coinz_df.to_sql('coins', engine, if_exists='replace', index=False)
    print(f"✓ Successfully loaded {len(coinz_df)} rows to 'coins' table")

except Exception as e:
    print(f"✗ Error loading data: {e}")



✓ Successfully loaded 5 rows to 'coins' table


In [128]:
pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.


In [137]:
import logging
import sys
import importlib
import load

load = importlib.reload(load)

from extract import extract_coins
from transform import transform_coins
from load import load_coins


logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')


def main():
    try:
        coin_list = extract_coins()
        coinz_df = transform_coins(coin_list)
        load_coins(coinz_df)
        print('ETL process completed successfully.')
    except Exception:
        logging.exception('ETL process failed')
        sys.exit(1)


if __name__ == "__main__":
    main()

✓ Successfully loaded 5 rows to 'coins' table
✓ Retrieved 5 rows from database
('Bitcoin', 'BTC', 20050338, '2010-07-17', 58656.3707)
('Ethereum', 'ETH', 120232214, '2015-08-07', 1576.9296)
('Tether', 'USDT', 190457881824, '2015-02-25', 0.9984)
ETL process completed successfully.
